In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import os
import datetime
import pandas as pd
import requests
from bs4 import BeautifulSoup
from time import sleep

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'IL BIS'

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(':','.')[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running IL BIS Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

regdict={
    regulatorName + ' 1': 'https://www.boi.org.il/en/economic-roles/supervision-and-regulation/list_supervised/',
}

Typology={
    regulatorName + ' 1': 'List of Supervised Banking Corporations and Merchant Acquirers',
}

In [4]:
#------------------------------------------------ Begin_Function ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key] = sqldict[key] + empty
    return sqldict


def get_soup(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36'
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    response.encoding = response.apparent_encoding
    return BeautifulSoup(response.text, 'html.parser')


def extract_supervised_entities(soup):
    records = []
    for heading in soup.find_all('h2'):
        category = heading.get_text(' ', strip=True)
        container = heading.find_next_sibling('div', class_='introOutroSpoilerContainer')
        if not container:
            continue

        for table in container.find_all('table'):
            for row in table.find_all('tr'):
                cells = [cell.get_text(' ', strip=True) for cell in row.find_all(['td', 'th'])]
                if len(cells) < 2:
                    continue

                reporting_symbol = cells[0]
                name = cells[1]
                if not reporting_symbol or reporting_symbol.lower() == 'reporting symbol' or not name:
                    continue

                records.append({
                    'category': category,
                    'reporting_symbol': reporting_symbol,
                    'name': name,
                })

    return records

In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------

for reg, url in regdict.items():
    list_name = Typology[reg]
    list_code = reg.split(' ')[-1]
    print(f"[INFO] : Working {reg} - {list_name}")

    soup = get_soup(url)
    records = extract_supervised_entities(soup)
    print(f"[INFO] : Found {len(records)} entities on {reg}")

    for record in records:
        sqldict['Name'].append(record['name'])
        sqldict['InternalID_1'].append(record['reporting_symbol'])
        sqldict['InternalID_1_type'].append('Reporting symbol')
        sqldict['Typology'].append(record['category'])
        sqldict['RegulationType'].append('Regulated')
        sqldict['RegCtry'].append('IL')
        sqldict['RegCode'].append('BIS')
        sqldict['ListCode'].append(list_code)
        sqldict['ListName'].append(list_name)
        sqldict['ListProcessDate'].append(processdate)

        sqldict = bourange_same_length_array(sqldict)

    sleep(1)

[INFO] : Working IL BIS 1 - List of Supervised Banking Corporations and Merchant Acquirers
[INFO] : Found 23 entities on IL BIS 1


In [6]:
#------------------------------------------------ Save DataFrame to Excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df.to_excel(filename, 'SQL Ready', index=False)
sleep(3)
print(f"[INFO] : Excel file '{filename}' saved successfully")

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_41548\1997204381.py:4: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)


[INFO] : Excel file 'IL BIS SQL Ready 2026-04-27 15.51.59.xlsx' saved successfully


In [ ]:
#------------------------------------------------ Data Integrity & Consistency Check ----------------------------------------

print('='*80)
print('DATA INTEGRITY & CONSISTENCY VERIFICATION')
print('='*80)

expected_categories = {'Banks', 'Foreign banks', 'Joint services companies', 'Merchant acquirers'}
actual_categories = set(df['CoType'].dropna().unique()) if not df.empty else set()
missing_categories = expected_categories - actual_categories

print('\n1. DATAFRAME SHAPE:')
print(f"   Total rows collected: {len(df)}")
print(f"   Total columns: {len(df.columns)}")

print('\n2. DATA DISTRIBUTION BY CATEGORY:')
if not df.empty:
    print(df.groupby('CoType').agg({'Name': 'count'}).rename(columns={'Name': 'Count'}))
else:
    print('   No data collected')

print('\n3. README CONSISTENCY:')
print('   RegCtry expected: IL')
print('   RegCode expected: BIS')
print('   ListCode expected: 1')
print(f"   Categories found: {sorted(actual_categories)}")

if missing_categories:
    print(f"   WARNING: Missing categories: {sorted(missing_categories)}")
else:
    print('   PASS: All expected categories found')

print('\n4. SAMPLE DATA:')
display_columns = ['Name', 'InternalID_1', 'InternalID_1_type', 'CoType', 'RegCtry', 'RegCode', 'ListCode', 'ListName']
print(df[display_columns].head(10).to_string(index=False))